# Task
Complete the fine-tuning of the model on the `ag_news` dataset, evaluate its performance, and document the entire process, including the training, evaluation results, and instructions for pushing the notebook to 'Agentic AI/Lab_Task/Fine_tuning_Lab_task1.ipynb' on GitHub.

# Fine-Tuning a Small Language Model (SLM) on Text Data

## Objective
The goal of this lab task is to fine-tune a Small Language Model (SLM) on a text dataset using Google Colab and evaluate its performance using appropriate metrics.

**Model Used:** distilgpt2  
**Dataset Used:** AG News dataset  

---

## Step 1 — Install Required Libraries

Install necessary libraries for model training and evaluation.

```python
!pip install transformers datasets accelerate evaluate -q


In [8]:
!pip install transformers datasets accelerate evaluate -q


## Step 2 — Load Dataset

In this step, we load the AG News dataset from Hugging Face. This dataset contains news articles grouped into categories and will be used to fine-tune the Small Language Model.

The dataset consists of:
- Title of the news article  
- Description of the news article  
- Label representing the category  

The dataset is divided into training and test splits. The training split is used to teach the model, and the test split is used later for evaluation. This confirms that the dataset is ready for preprocessing.


In [9]:
from datasets import load_dataset

dataset = load_dataset("sh0416/ag_news")
dataset


README.md: 0.00B [00:00, ?B/s]

train.jsonl:   0%|          | 0.00/33.7M [00:00<?, ?B/s]

test.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['label', 'title', 'description'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['label', 'title', 'description'],
        num_rows: 7600
    })
})

## Step 3 — Inspect Dataset Structure

In this step, we examine the structure of the dataset to understand how the data is organized. Inspecting a sample helps identify the available fields and decide how to prepare the data for training.

We observe that the dataset contains separate fields for the news title and description. Since language models require continuous text input, these fields will later be combined into a single text column.

This step ensures we clearly understand the dataset format before preprocessing.


In [10]:
dataset["train"][0]


{'label': 3,
 'title': 'Wall St. Bears Claw Back Into the Black (Reuters)',
 'description': "Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again."}

## Step 4 — Merge Text Fields

The dataset contains two separate text-related fields: title and description. For language model training, the model requires a single continuous text input.

In this step, we combine the title and description into one new field called "text". This provides richer contextual information to the model and improves learning quality.

This preprocessing step prepares the dataset for tokenization.


In [11]:
def combine_text(example):
    example["text"] = example["title"] + " " + example["description"]
    return example

dataset = dataset.map(combine_text)


Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

## Step 5 — Load Model and Tokenizer

In this step, we load a Small Language Model (SLM) and its tokenizer. The model used is distilgpt2, which is a lightweight version of GPT-2 and suitable for training in environments like Google Colab.

The tokenizer converts text into numerical tokens that the model can understand. The pad token is set to the end-of-sequence token to ensure proper handling of fixed-length inputs.

This step prepares the model and tokenizer for text processing and training.


In [12]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

tokenizer.pad_token = tokenizer.eos_token


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

## Step 6 — Tokenization

Language models cannot understand raw text. Therefore, the combined text must be converted into numerical tokens using the tokenizer.

Tokenization transforms sentences into token IDs and creates attention masks so the model knows which parts of the input are meaningful. The text is truncated or padded to a fixed length to ensure consistent input size during training.

This step prepares the dataset in a numerical format suitable for the model.


In [13]:
def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

tokenized_datasets = dataset.map(tokenize_function, batched=True)


Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

## Step 7 — Add Labels for Language Modeling

For causal language modeling, the model learns to predict the next word in a sequence. To enable this, the labels must be the same as the input token IDs.

In this step, labels are added to the dataset so the model can compute the training loss. Unnecessary columns such as title, description, and label are removed since they are not required for language model training.

The dataset format is then converted to PyTorch tensors for efficient training.


In [14]:
tokenized_datasets = tokenized_datasets.map(
    lambda x: {"labels": x["input_ids"]},
    batched=True
)

tokenized_datasets = tokenized_datasets.remove_columns(
    ["title", "description", "text", "label"]
)

tokenized_datasets.set_format("torch")


Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

## Step 8 — Training Setup

In this step, we define the training configuration using TrainingArguments. These parameters control how the model learns.

Key settings include:
- Learning rate for weight updates
- Batch size for training and evaluation
- Number of training epochs
- Weight decay to prevent overfitting
- Logging and model saving strategy

This step prepares the training process before fine-tuning begins.


In [16]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",   # <- FIXED
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_dir="./logs",
    save_strategy="epoch",
    report_to="none"
)


`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


## Step 9 — Train the Model

In this step, the model is fine-tuned using the Trainer API. The training dataset is used to update model weights, while the test dataset is used for evaluation at the end of each epoch.

During training, the model learns patterns from the news text and adapts its internal parameters to better predict the next word in a sequence.

This step performs the actual fine-tuning of the Small Language Model.


In [19]:
small_train = tokenized_datasets["train"].shuffle(seed=42).select(range(20000))
small_test = tokenized_datasets["test"].shuffle(seed=42).select(range(4000))


In [20]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train,
    eval_dataset=small_test
)

trainer.train()


Epoch,Training Loss,Validation Loss
1,1.470883,1.381030
2,1.419280,1.372242


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2500, training_loss=1.4471419189453125, metrics={'train_runtime': 917.7535, 'train_samples_per_second': 43.585, 'train_steps_per_second': 2.724, 'total_flos': 1306483752960000.0, 'train_loss': 1.4471419189453125, 'epoch': 2.0})

## Step 10 — Model Evaluation

After training, the model is evaluated using the test dataset. Since this task involves language modeling, performance is measured using loss and perplexity.

Loss shows how much error the model makes while predicting the next word. Perplexity indicates how confused the model is. A lower perplexity value means the model has learned language patterns better.

This step helps assess the effectiveness of fine-tuning.


In [21]:
import math

results = trainer.evaluate()

perplexity = math.exp(results["eval_loss"])

print("Evaluation Loss:", results["eval_loss"])
print("Perplexity:", perplexity)


Evaluation Loss: 1.3722422122955322
Perplexity: 3.9441844871041947


## Step 11 — Text Generation Test

In this step, we test the fine-tuned model by generating text. A short prompt is given, and the model predicts the continuation based on what it learned during training.

This helps us observe whether the model has adapted to the news-style text patterns from the dataset.


In [24]:
import torch

input_text = "Breaking news in technology:"
inputs = tokenizer(input_text, return_tensors="pt")

# MOVE INPUTS TO SAME DEVICE AS MODEL
inputs = {k: v.to(model.device) for k, v in inputs.items()}

outputs = model.generate(
    **inputs,
    max_length=60,
    num_return_sequences=1
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Breaking news in technology: Microsoft #39;s new Windows XP operating system is a good thing, but it's not the only thing Microsoft is doing.


## Final Results

The Small Language Model was successfully fine-tuned on the AG News dataset. Training loss decreased across epochs, indicating that the model learned patterns from the dataset.

The evaluation showed a reduction in perplexity, meaning the model became less confused while predicting the next word. This confirms that fine-tuning improved its language modeling capability.

Text generation tests showed that the model produces more structured, news-style sentences compared to the base model.

## Conclusion

This task demonstrated how a pre-trained Small Language Model can be adapted to a specific domain using fine-tuning. The experiment highlighted the role of tokenization, training configuration, and evaluation metrics such as perplexity in assessing performance.

Fine-tuning allows general models to specialize in domain-specific language patterns effectively.
